# 🎙️ YouTube Hindi / English Transcriber (free, Google Colab)

Paste YouTube links → get transcripts (TXT, timestamps, SRT). Runs on Google's free GPU with **Whisper large-v3**, good for Hindi, English and Hinglish, and handles hours of audio.

**How to use (works on iPad Safari):**
1. Tap ▶ on **Step 1** (once per session, ~1–2 min). If asked *"Run anyway?"* tap **Run anyway**.
2. Paste your link(s) in **Step 2**, pick the language, tap ▶.
3. Transcripts appear below and download as a ZIP (or go to Google Drive if you tick that box).

**If YouTube blocks the download** ("Sign in to confirm you're not a bot"): try **Runtime → Disconnect and delete runtime** and run Step 1 + 2 again (new machine). If it still fails, save the videos' audio to your iPad or Google Drive and use **Step 3**, which transcribes uploaded files and always works.

Keep this tab open while it runs. Web app version: https://youtube-transcript-live.streamlit.app

In [ ]:
#@title **Step 1 — Setup** (run once per session)
import subprocess, sys, os, shutil
print("Installing tools…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "deno", "faster-whisper"], check=True)
if os.path.isdir("/content/youtube-transcriber"):
    subprocess.run(["git", "-C", "/content/youtube-transcriber", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/agshivam/youtube-transcriber", "/content/youtube-transcriber"], check=True)
sys.path.insert(0, "/content/youtube-transcriber")
import importlib, transcriber
importlib.reload(transcriber)
gpu = shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none (slower). For speed: Runtime → Change runtime type → T4 GPU, then run Step 1 again.")
print("FFmpeg:", "ok" if shutil.which("ffmpeg") else "missing")
print("✅ Setup done. Now run Step 2.")

In [ ]:
#@title **Step 2 — Paste YouTube link(s) and run**
links = "https://youtu.be/Jt-cZQjUdaY" #@param {type:"string"}
language = "Hindi" #@param ["Hindi", "English", "Auto (Hindi / English / Hinglish)"]
model_size = "large-v3" #@param ["large-v3", "large-v3-turbo", "medium", "small"]
use_youtube_captions_if_available = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown Separate several links with spaces, commas or new lines. Leave *use YouTube captions* off for the best Hindi quality (YouTube's auto-captions are often poor).

import re, sys, time, tempfile, zipfile, shutil
from pathlib import Path
sys.path.insert(0, "/content/youtube-transcriber")
import transcriber as T

LANG = {"Hindi": "hi", "English": "en"}.get(language)
OUT = Path("/content/transcripts"); OUT.mkdir(parents=True, exist_ok=True)

def get_model():
    global _MODEL, _MODEL_NAME, _RUNTIME
    if globals().get("_MODEL_NAME") != model_size:
        print(f"Loading Whisper {model_size} (first time downloads ~1.5 GB)…")
        _MODEL, _RUNTIME = T.load_whisper_model(model_size)
        _MODEL_NAME = model_size
        print("Model ready on", _RUNTIME)
    return _MODEL

def whisper_transcribe(path, title):
    model = get_model()
    gpu = "GPU" in _RUNTIME
    segments, info = model.transcribe(T.load_audio_array(path), language=LANG, beam_size=5, vad_filter=True,
                                      vad_parameters={"min_silence_duration_ms": 500},
                                      condition_on_previous_text=True)
    total = max(float(getattr(info, "duration", 0) or 0), 1.0)
    done, last = [], 0.0
    started = time.time()
    for seg in segments:
        done.append(seg)
        pct = min(seg.end / total * 100, 100)
        if pct - last >= 5 or pct >= 100:
            print(f"   {pct:5.1f}%  ({T.format_short_timestamp(seg.end)} of {T.format_short_timestamp(total)}, {time.time()-started:.0f}s)")
            last = pct
    return T.build_outputs(title, done, getattr(info, "language", LANG), source=f"Whisper {model_size} on {_RUNTIME}")

def transcribe_link(url):
    notes = []
    if use_youtube_captions_if_available:
        result = T.fetch_youtube_captions(url, LANG, notes)
        for n in notes: print("   " + n)
        if result is not None and (LANG is None or result.detected_language == LANG):
            return result
    with tempfile.TemporaryDirectory() as tmp:
        print("   Downloading audio…")
        audio, title = T.download_youtube_audio(url, tmp)
        print(f"   Got audio: {title}")
        return whisper_transcribe(audio, title)

urls = [u for u in re.split(r"[\s,]+", links.strip()) if u]
if not urls:
    raise SystemExit("Paste at least one YouTube link above.")

saved, failed = [], []
for i, url in enumerate(urls, 1):
    print(f"\n[{i}/{len(urls)}] {url}")
    try:
        r = transcribe_link(url)
    except T.TranscriptionError as exc:
        msg = str(exc)
        if T._is_blocked(exc):
            msg += ("\n   → YouTube blocked this Colab machine. Either: Runtime → Disconnect and delete runtime, then Step 1 + Step 2 again;"
                    "\n     or save the audio to your iPad / Google Drive and use Step 3 (works every time).")
        print("   ❌", msg); failed.append(url); continue
    for name, text in ((r.txt_filename, r.clean_text), (r.timestamped_filename, r.timestamped_text), (r.srt_filename, r.srt_text)):
        (OUT / name).write_text(text, encoding="utf-8")
        saved.append(OUT / name)
    print(f"   ✅ {r.title} — {r.source}, language: {r.detected_language}")
    print("   ---- transcript ----")
    print(r.clean_text)
    print("   --------------------")

if saved:
    if save_to_google_drive:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/YouTube Transcripts"); dest.mkdir(parents=True, exist_ok=True)
        for f in saved: shutil.copy(f, dest / f.name)
        print(f"\n📁 Saved {len(saved)} files to Google Drive → MyDrive/YouTube Transcripts")
    zip_path = Path("/content/transcripts.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for f in saved: z.write(f, f.name)
    try:
        from google.colab import files
        files.download(str(zip_path))
        print("⬇️ Downloading transcripts.zip (also in the 📁 Files panel on the left → transcripts).")
    except Exception:
        print("Files are in /content/transcripts")
print(f"\nDone: {len(urls) - len(failed)} succeeded, {len(failed)} failed.")

In [ ]:
#@title **Step 3 — YouTube blocked? Transcribe audio/video files instead**
source = "Upload from this device" #@param ["Upload from this device", "Google Drive folder"]
drive_folder = "YouTube Audio" #@param {type:"string"}
language = "Hindi" #@param ["Hindi", "English", "Auto (Hindi / English / Hinglish)"]
model_size = "large-v3-turbo" #@param ["large-v3", "large-v3-turbo", "medium", "small"]
save_to_google_drive = True #@param {type:"boolean"}
#@markdown **Upload**: after tapping ▶, tap *Choose files* below and select one or many files (mp3, m4a, mp4, wav…).<br>**Google Drive folder**: put the files in *My Drive → YouTube Audio* first, then run. Results go to *My Drive → YouTube Transcripts*.

import sys, time, zipfile, shutil
from pathlib import Path
sys.path.insert(0, "/content/youtube-transcriber")
import transcriber as T

LANG = {"Hindi": "hi", "English": "en"}.get(language)
OUT = Path("/content/transcripts"); OUT.mkdir(parents=True, exist_ok=True)
MEDIA = {".mp3", ".m4a", ".mp4", ".wav", ".ogg", ".opus", ".webm", ".aac", ".flac", ".mov", ".mkv", ".mpeg", ".mpga", ".3gp"}

def get_model():
    global _MODEL, _MODEL_NAME, _RUNTIME
    if globals().get("_MODEL_NAME") != model_size:
        print(f"Loading Whisper {model_size} (first time downloads up to ~3 GB, 1–2 min)…")
        _MODEL, _RUNTIME = T.load_whisper_model(model_size)
        _MODEL_NAME = model_size
        print("Model ready on", _RUNTIME)
    return _MODEL

def whisper_transcribe(path, title):
    model = get_model()
    segments, info = model.transcribe(T.load_audio_array(path), language=LANG, beam_size=5, vad_filter=True,
                                      vad_parameters={"min_silence_duration_ms": 500},
                                      condition_on_previous_text=True)
    total = max(float(getattr(info, "duration", 0) or 0), 1.0)
    done, last, started = [], 0.0, time.time()
    for seg in segments:
        done.append(seg)
        pct = min(seg.end / total * 100, 100)
        if pct - last >= 10 or pct >= 100:
            print(f"   {pct:5.1f}%  ({T.format_short_timestamp(seg.end)} of {T.format_short_timestamp(total)}, {time.time()-started:.0f}s)")
            last = pct
    return T.build_outputs(title, done, getattr(info, "language", LANG), source=f"Whisper {model_size} on {_RUNTIME}")

drive_mounted = False
if source == "Google Drive folder" or save_to_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_mounted = True

if source == "Google Drive folder":
    folder = Path("/content/drive/MyDrive") / drive_folder
    if not folder.is_dir():
        raise SystemExit(f"Folder not found: My Drive → {drive_folder}. Create it and put your audio files in it.")
    inputs = sorted(p for p in folder.iterdir() if p.suffix.lower() in MEDIA)
else:
    from google.colab import files
    up_dir = Path("/content/uploads"); up_dir.mkdir(exist_ok=True)
    uploaded = files.upload()
    inputs = []
    for name, data in uploaded.items():
        p = up_dir / name
        p.write_bytes(data)
        inputs.append(p)
    inputs = [p for p in inputs if p.suffix.lower() in MEDIA]
if not inputs:
    raise SystemExit("No audio/video files found.")

dest = None
if save_to_google_drive and drive_mounted:
    dest = Path("/content/drive/MyDrive/YouTube Transcripts"); dest.mkdir(parents=True, exist_ok=True)

saved, failed = [], []
for i, path in enumerate(inputs, 1):
    print(f"\n[{i}/{len(inputs)}] {path.name}")
    try:
        r = whisper_transcribe(path, T.safe_stem(path.stem) or "Transcript")
    except Exception as exc:
        print("   ❌", exc); failed.append(path.name); continue
    for name, text in ((r.txt_filename, r.clean_text), (r.timestamped_filename, r.timestamped_text), (r.srt_filename, r.srt_text)):
        (OUT / name).write_text(text, encoding="utf-8"); saved.append(OUT / name)
        if dest: shutil.copy(OUT / name, dest / name)
    print(f"   ✅ done ({r.detected_language})" + (" → saved to Google Drive" if dest else ""))
    print("   ---- transcript ----"); print(r.clean_text); print("   --------------------")

if saved:
    zip_path = Path("/content/transcripts.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for f in saved: z.write(f, f.name)
    try:
        from google.colab import files as _f
        _f.download(str(zip_path))
    except Exception:
        pass
    print("\n⬇️ All transcripts: transcripts.zip" + (" and My Drive → YouTube Transcripts" if dest else ""))
print(f"\nDone: {len(inputs) - len(failed)} succeeded, {len(failed)} failed.")